In [ ]:
import re
import json
import joblib
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import matplotlib.pyplot as plt


In [ ]:
DATA_PATH = "listings.csv"   # or "listings.csv.gz"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['price'] = (
    df['price']
    .str.replace('$','', regex=False)
    .str.replace('€','', regex=False)
    .str.replace(',','', regex=False)
    .astype(float)
)


In [ ]:
df['price'].head()


In [ ]:
df.isna().mean().sort_values(ascending=False).head(20)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df['price'], bins=50)
plt.xlim(0, 500)  # avoid extreme outliers
plt.title("Distribution of Airbnb Prices in Barcelona")
plt.show()


In [ ]:
df.shape

In [ ]:
sns.scatterplot(data=df, x='number_of_reviews', y='price')
plt.xlim(0,200)
plt.show()


In [ ]:
sns.scatterplot(data=df, x='review_scores_rating', y='price')


In [ ]:
sns.scatterplot(data=df, x='longitude', y='latitude', hue='price', size=1, alpha=0.4)
plt.title("Airbnb Prices Across Barcelona")
plt.show()


In [ ]:
DROP_COLS = [
    "id", "listing_url", "scrape_id", "source",
    "picture_url", "host_url", "host_thumbnail_url", "host_picture_url",
    "last_scraped", "calendar_last_scraped", "calendar_updated",

    # long text fields (NLP not in scope)
    "name", "description", "neighborhood_overview", "host_about",

    # redundant neighborhood fields
    "neighbourhood", "neighbourhood_group_cleansed", "host_neighbourhood",

    # leakage / derived
    "estimated_revenue_l365d", "estimated_occupancy_l365d",

    # post-hoc / date-y reviews
    "first_review", "last_review",

    # messy/high-cardinality and sparse
    "license",
]


In [ ]:
def clean_price_to_float(s: pd.Series) -> pd.Series:
    # Works for "€120.00", "$120.00", "120", "120.0", etc.
    return (
        s.astype(str)
         .str.replace(r"[\€, \$]", "", regex=True)
         .str.replace(",", "", regex=False)
         .replace("nan", np.nan)
         .astype(float)
    )

def percent_to_float(s: pd.Series) -> pd.Series:
    # "95%" -> 95.0 ; missing stays NaN
    return (
        s.astype(str)
         .str.replace("%", "", regex=False)
         .replace("nan", np.nan)
         .astype(float)
    )

def parse_bathrooms_text(x) -> float:
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    m = re.search(r"(\d+(\.\d+)?)", x)
    return float(m.group(1)) if m else np.nan

def amenities_count(x) -> float:
    if pd.isna(x):
        return np.nan
    # InsideAirbnb amenities often look like {"Wifi","Kitchen",...}
    # Counting quotes is a simple robust approximation.
    s = str(x)
    return s.count('"')

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


In [ ]:
# Reference points in Barcelona (approx.)
CATALUNYA = (41.3870, 2.1700)      # Plaça de Catalunya
BARCELONETA = (41.3780, 2.1920) 
def engineer_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # Drop policy columns if present
    cols_to_drop = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=cols_to_drop, errors="ignore")

    # Bathrooms / amenities
    if "bathrooms_text" in df.columns:
        df["bathrooms_num"] = df["bathrooms_text"].apply(parse_bathrooms_text)
    if "amenities" in df.columns:
        df["amenities_count"] = df["amenities"].apply(amenities_count)

    # Percent -> numeric
    if "host_response_rate" in df.columns:
        df["host_response_rate_num"] = percent_to_float(df["host_response_rate"])
    if "host_acceptance_rate" in df.columns:
        df["host_acceptance_rate_num"] = percent_to_float(df["host_acceptance_rate"])

    # Host tenure
    if "host_since" in df.columns:
        hs = pd.to_datetime(df["host_since"], errors="coerce")
        ref = pd.Timestamp.today(tz=None).normalize()
        df["host_tenure_days"] = (ref - hs).dt.days

    # Distances
    if "latitude" in df.columns and "longitude" in df.columns:
        df["dist_to_center_km"] = haversine_km(df["latitude"], df["longitude"], *CATALUNYA)
        df["dist_to_beach_km"]  = haversine_km(df["latitude"], df["longitude"], *BARCELONETA)

    return df


def prepare_training_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = engineer_features(df_raw)

    # Target cleaning only here (training only)
    df["price"] = clean_price_to_float(df["price"])
    df = df.dropna(subset=["price"])
    df = df[df["price"] > 0]
    df["log_price"] = np.log1p(df["price"])

    return df


In [ ]:
df_prep = prepare_training_dataset(df)

print("After prep:", df_prep.shape)
df_prep[["price", "log_price"]].describe()


In [ ]:
# Price distribution (cap for readability)
plt.figure()
df_prep["price"].clip(upper=500).hist(bins=50)
plt.title("Nightly price distribution (clipped at €500)")
plt.xlabel("Price (€)")
plt.ylabel("Count")
plt.show()

plt.figure()
df_prep["log_price"].hist(bins=50)
plt.title("log1p(price) distribution")
plt.xlabel("log_price")
plt.ylabel("Count")
plt.show()

# Room type mean price
if "room_type" in df_prep.columns:
    room_means = df_prep.groupby("room_type")["price"].mean().sort_values()
    plt.figure()
    room_means.plot(kind="bar")
    plt.title("Average price by room_type")
    plt.ylabel("Mean price (€)")
    plt.show()


In [ ]:
TARGET = "log_price"

NUM_FEATURES = [
    "accommodates", "bedrooms", "beds", "bathrooms_num",
    "minimum_nights", "maximum_nights",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "reviews_per_month",
    "review_scores_rating",
    "amenities_count",
    "dist_to_center_km", "dist_to_beach_km",
    "host_tenure_days",
    "host_listings_count",
    "host_response_rate_num", "host_acceptance_rate_num",
]

CAT_FEATURES = [
    "room_type", "property_type", "neighbourhood_cleansed",
    "instant_bookable", "has_availability",
    "host_is_superhost", "host_identity_verified", "host_has_profile_pic",
    "host_response_time",
]

# Keep only columns that exist in your dataframe
NUM_FEATURES = [c for c in NUM_FEATURES if c in df_prep.columns]
CAT_FEATURES = [c for c in CAT_FEATURES if c in df_prep.columns]

FEATURES = NUM_FEATURES + CAT_FEATURES
print("Numeric:", NUM_FEATURES)
print("Categorical:", CAT_FEATURES)
print("Total features:", len(FEATURES))


In [ ]:
X = df_prep[FEATURES]
y = df_prep[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUM_FEATURES),
        ("cat", categorical_transformer, CAT_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=0.0  # <--- force dense output
    
)

ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge(alpha=2.0, random_state=42)),
])

hgb_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingRegressor(
        random_state=42,
        max_depth=None,
        learning_rate=0.08,
        max_leaf_nodes=31
    )),
])


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_report(pipe, X, y, name: str):
    scoring = {
        "r2": "r2",
        "neg_mae": "neg_mean_absolute_error",
        "neg_rmse": "neg_root_mean_squared_error"
    }
    res = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"\n{name}")
    print("R2 mean:", round(res["test_r2"].mean(), 4), "±", round(res["test_r2"].std(), 4))
    print("MAE(log) mean:", round((-res["test_neg_mae"]).mean(), 4))
    print("RMSE(log) mean:", round((-res["test_neg_rmse"]).mean(), 4))

cv_report(ridge_pipe, X_train, y_train, "Ridge (baseline)")
cv_report(hgb_pipe, X_train, y_train, "HistGradientBoosting (strong)")


In [ ]:
from sklearn.linear_model import ElasticNet

enet_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", ElasticNet(alpha=0.001, l1_ratio=0.2, random_state=42)),
])


In [ ]:
cv_report(enet_pipe, X_train, y_train, "ElasticNet ")


In [ ]:
best_pipe = hgb_pipe
best_pipe.fit(X_train, y_train)

pred_log = best_pipe.predict(X_test)

mae_log = mean_absolute_error(y_test, pred_log)
rmse_log = mean_squared_error(y_test, pred_log, squared=False)
r2 = r2_score(y_test, pred_log)

# Convert back to euros (for audience impact)
pred_eur = np.expm1(pred_log)
true_eur = np.expm1(y_test)

mae_eur = np.mean(np.abs(true_eur - pred_eur))
rmse_eur = np.sqrt(np.mean((true_eur - pred_eur)**2))

print("TEST METRICS (HGB)")
print("R2:", round(r2, 4))
print("MAE(log):", round(mae_log, 4), " | RMSE(log):", round(rmse_log, 4))
print("MAE (€):", round(mae_eur, 2), " | RMSE (€):", round(rmse_eur, 2))


In [ ]:
plt.figure()
plt.scatter(true_eur, pred_eur, alpha=0.25)
plt.xlabel("True price (€)")
plt.ylabel("Predicted price (€)")
plt.title("Predicted vs True Nightly Price (Test Set)")
plt.xlim(0, 500); plt.ylim(0, 500)
plt.show()


In [ ]:
median_ae = np.median(np.abs(true_eur - pred_eur))
print("Median Absolute Error (€):", round(median_ae, 2))


In [ ]:
perm = permutation_importance(
    best_pipe, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

# Convert to % contribution
importance_pct = 100 * (importance / importance.sum())
imp_df = pd.DataFrame({
    "importance_drop": importance,
    "importance_pct": importance_pct
}).sort_values("importance_pct", ascending=False)

imp_df.head(20)


In [ ]:
top_n = 15
plt.figure()
imp_df.head(top_n).sort_values("importance_pct").plot(kind="barh", y="importance_pct", legend=False)
plt.title(f"Top {top_n} Global Drivers of Price (Permutation Importance %)")
plt.xlabel("Importance (%)")
plt.ylabel("")
plt.show()


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Pick top numeric features that exist
top_numeric_candidates = [c for c in imp_df.index if c in NUM_FEATURES]
pdp_features = top_numeric_candidates[:4]
print("PDP features:", pdp_features)

PartialDependenceDisplay.from_estimator(best_pipe, X_test, features=pdp_features)
plt.show()


In [ ]:
PartialDependenceDisplay.from_estimator(best_pipe, X_test, features=["dist_to_center_km"])
plt.show()


In [ ]:
# Clean anomaly analysis dataset
test_out = df_prep.loc[X_test.index].copy()
test_out["true_price"] = np.expm1(y_test)
test_out["pred_price"] = np.expm1(pred_log)

# Filters for realism
test_out = test_out[
    (test_out["true_price"] <= 500) &
    (test_out["pred_price"] <= 500) &
    (test_out["pred_price"] >= 30)
]

test_out["diff_eur"] = test_out["true_price"] - test_out["pred_price"]
test_out["pct_diff"] = 100 * test_out["diff_eur"] / test_out["pred_price"]

test_out["pricing_flag"] = np.where(
    test_out["pct_diff"] > 30, "Overpriced",
    np.where(test_out["pct_diff"] < -30, "Underpriced", "Fair")
)

test_out["pricing_flag"].value_counts()


In [ ]:
display_cols = [c for c in ["neighbourhood_cleansed","room_type","property_type","accommodates",
                            "bedrooms","bathrooms_num","amenities_count","dist_to_center_km"] if c in test_out.columns]

print("\nMost Overpriced (relative to model):")
test_out.sort_values("pct_diff", ascending=False)[display_cols + ["true_price","pred_price","pct_diff"]].head(10)



In [ ]:
print("\nMost Underpriced (relative to model):")
test_out.sort_values("pct_diff", ascending=True)[display_cols + ["true_price","pred_price","pct_diff"]].head(10)


In [ ]:
test_out = df_prep.loc[X_test.index].copy()
test_out["true_price"] = np.expm1(y_test)
test_out["pred_price"] = np.expm1(pred_log)

nb = (
    test_out
    .groupby("neighbourhood_cleansed")[["true_price","pred_price"]]
    .median()
    .sort_values("true_price", ascending=False)
)

nb["median_true_minus_pred"] = nb["true_price"] - nb["pred_price"]
nb.head(15)


In [ ]:
if "neighbourhood_cleansed" in X_test.columns:
    top_nb = nb.head(15).sort_values("true_price")
    plt.figure()
    top_nb["true_price"].plot(kind="barh")
    plt.title("Top 15 Neighborhoods by Median Nightly Price (Test Set)")
    plt.xlabel("Median price (€)")
    plt.ylabel("")
    plt.show()


In [ ]:
from sklearn.metrics import r2_score

rows = []
for name, m in fitted.items():
    pred_log = m.predict(X_test)
    r2 = r2_score(y_test, pred_log)

    pred_eur = np.expm1(pred_log)
    mae_eur = np.mean(np.abs(true_eur - pred_eur))
    medae_eur = np.median(np.abs(true_eur - pred_eur))

    rows.append([name, r2, mae_eur, medae_eur])

score_df = pd.DataFrame(rows, columns=["Model", "R2", "MAE (€)", "Median AE (€)"]).sort_values("R2", ascending=False)
score_df


In [ ]:
plt.figure()
plt.bar(score_df["Model"], score_df["R2"])
plt.ylabel("R² score")
plt.title("Model Performance Comparison (Test Set)")
plt.ylim(0, 1)
plt.show()


In [ ]:
num_df = df_prep[NUM_FEATURES + ["price"]].corr()

plt.figure(figsize=(10, 8))
plt.imshow(num_df, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(num_df)), num_df.columns, rotation=90)
plt.yticks(range(len(num_df)), num_df.columns)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()


In [ ]:
plt.figure()
plt.scatter(df_prep["dist_to_center_km"], df_prep["price"], alpha=0.3)
plt.xlabel("Distance to city center (km)")
plt.ylabel("Nightly price (€)")
plt.title("Price vs Distance to City Center")
plt.ylim(0, 500)
plt.show()


In [ ]:
from sklearn.base import clone

models = {
    "Ridge": ridge_pipe,
    "ElasticNet": enet_pipe,
    "HGB": hgb_pipe,
}

# Fit models
fitted = {}
for name, pipe in models.items():
    m = clone(pipe)
    m.fit(X_train, y_train)
    fitted[name] = m

# Predictions in euros
true_eur = np.expm1(y_test.values)
order = np.argsort(true_eur)

plt.figure(figsize=(12, 5))
plt.plot(true_eur[order], label="True price")

for name, m in fitted.items():
    pred_log = m.predict(X_test)
    pred_eur = np.expm1(pred_log)
    plt.plot(pred_eur[order], label=f"{name} predicted", alpha=0.9)

plt.title("True vs Predicted Nightly Price (sorted by true price)")
plt.xlabel("Test listings (sorted)")
plt.ylabel("Price (€)")
plt.ylim(0, 500)  # keep readable
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))

for name, m in fitted.items():
    pred_eur = np.expm1(m.predict(X_test))
    residual = (true_eur - pred_eur)[order]
    plt.plot(residual, label=f"{name} residual", alpha=0.9)

plt.axhline(0, linewidth=1)
plt.title("Residuals (True - Predicted), sorted by true price")
plt.xlabel("Test listings (sorted)")
plt.ylabel("Residual (€)")
plt.ylim(-300, 300)
plt.legend()
plt.show()


In [ ]:
import json, joblib

MODEL_PATH = "airbnb_barcelona_price_model.joblib"
META_PATH = "airbnb_barcelona_model_metadata.json"

joblib.dump(best_pipe, MODEL_PATH)

metadata = {
    "target": "log_price",
    "features": FEATURES,
    "num_features": NUM_FEATURES,
    "cat_features": CAT_FEATURES,
    "notes": "Model predicts log1p(price); convert back with expm1()."
}
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", MODEL_PATH, META_PATH)


In [ ]:
IMP_PATH = "permutation_importance.csv"
imp_df.to_csv(IMP_PATH, index=True)
print("Saved:", IMP_PATH)


In [ ]:
import folium

# sample to keep it fast
plot_df = df_prep.dropna(subset=["latitude","longitude","price"]).sample(min(3000, len(df_prep)), random_state=42)

m = folium.Map(location=[plot_df["latitude"].mean(), plot_df["longitude"].mean()], zoom_start=12, tiles="OpenStreetMap")

# clip price for stable colors
p = plot_df["price"].clip(upper=300)

for lat, lon, price in zip(plot_df["latitude"], plot_df["longitude"], p):
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        fill=True,
        fill_opacity=0.5,
        popup=f"€{price:.0f}/night",
    ).add_to(m)

m
